# Phase 5 — Neural Network (Final Model)
Competition: Playground Series S6E7 — Predicting Student Health Risk  
Metric: **Balanced accuracy**  
Model: Keras MLP — 3-class softmax + class_weight compensation

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import joblib
import tensorflow as tf

from src.data_prep import load_data, split_data, get_column_types
from src.features import load_preprocessor
from src.train import train_neural_net
from src.evaluate import evaluate_keras, plot_confusion_matrix, plot_training_history

## 1. Load data + fitted preprocessor (already saved in notebook 02)

In [ ]:
train_df, test_df = load_data('../data/raw')
X_train, X_val, y_train, y_val = split_data(train_df)

# Load preprocessor fitted in notebook 02 — never re-fit
preprocessor = load_preprocessor('../models/preprocessor.pkl')
X_train_proc = preprocessor.transform(X_train)
X_val_proc   = preprocessor.transform(X_val)

print('X_train_proc shape:', X_train_proc.shape)

## 2. Train neural network

In [ ]:
nn, le, history = train_neural_net(
    X_train_proc, y_train,
    X_val_proc,   y_val,
    epochs=20, batch_size=2048,
    save_path='../models/model_nn.h5'
)

# Save label encoder — needed for decoding predictions
joblib.dump(le, '../models/label_encoder.pkl')
print('Label classes:', le.classes_)

## 3. Evaluate with balanced accuracy (not Keras accuracy)

In [ ]:
y_val_enc = le.transform(y_val)
nn_bal_acc, nn_preds_enc = evaluate_keras(nn, le, X_val_proc, y_val_enc, 'Neural Net MLP')
plot_confusion_matrix(y_val_enc, nn_preds_enc, le.classes_, title='NN Confusion Matrix')

## 4. Training curves

In [ ]:
plot_training_history(history)

## 5. Compare all models — pick the winner

In [ ]:
import pandas as pd

prev = pd.read_csv('../models/results_so_far.csv')
nn_row = pd.DataFrame([{
    'Model': 'Neural Net (MLP, class-weighted)',
    'Val balanced_acc': nn_bal_acc,
    'CV mean': '-',
    'CV std': '-',
    'Notes': f'{len(history.history["loss"])} epochs, early stopping'
}])

all_results = pd.concat([prev, nn_row], ignore_index=True)
print(all_results.to_markdown(index=False))

In [ ]:
# Append NN result to experiments.md (preserves existing V4 data)
with open('../experiments.md', 'a') as f:
    f.write('\n## V1-V2 (from notebook 03 run)\n\n')
    f.write('| Model | Val balanced accuracy | Notes |\n')
    f.write('|---|---|---|\n')
    for _, row in all_results.iterrows():
        f.write(f"| {row['Model']} | {row['Val balanced_acc']} | {row['Notes']} |\n")
print('NN result appended to experiments.md.')

## 6. Promote best model to model_final

> **Decision:** Neural Net (MLP) wins with balanced accuracy 0.9006 vs RF 0.8790 vs LR 0.8574.  
> `models/model_final.pkl` is set to the Neural Net model.

In [ ]:
import shutil

# NN wins: 0.9006 > RF 0.8790 > LR 0.8574
winner = 'nn'

if winner == 'nn':
    shutil.copy('../models/model_nn.h5', '../models/model_final.h5')
    print('Winner: Neural Net — model_final.h5 is the submission model.')
    print('Use predict_keras() in src/predict.py to generate submission.')
else:
    print('Winner: Random Forest — model_final.pkl already set in notebook 02.')
    print('Use predict_sklearn() in src/predict.py to generate submission.')